# Day 2 | ILT 2: Lakeflow Connect + Storage Credentials & External Locations
### GlobalMart Data Engineering Bootcamp
---
**Session Time:** Day 2, Late Morning (90 min)
**Comes after:** ILT 1 — CDC Concepts (WAL & Logical Replication)
**Comes before:** HOL 1 — hands-on: create your own storage credential + external location, and stand up a Lakeflow Connect pipeline

---
### What We Cover Today
1. Recap — why watermark/JDBC polling doesn't scale, and what Lakeflow Connect replaces it with
2. How Lakeflow Connect works end to end (connection → pipeline → destination table), and its two connector modes (log-based vs query-based)
3. Worked example — the Supabase-side CDC setup, plus GlobalMart's real `ecom_gbmart_conn` connection and the actual `orders_data_ingestion_cdc` pipeline for `orders` / `order_items`
4. GlobalMart's full ingestion tool assignment (Lakeflow Connect vs Autoloader, table by table)
5. Old approach: mounting ADLS with `dbutils.fs.mount()` — and why it's no longer used
6. Why hardcoded storage keys are also a governance problem
7. The two Unity Catalog building blocks: **Storage Credential** and **External Location**
8. Worked example — GlobalMart's real `gbmart-ext-loc` location

> **Instructor note:** 90 minutes. ~15 min recap + motivation, ~30 min Lakeflow Connect live demo (UI walkthrough on `ecom_gbmart_conn` / `orders_data_ingestion_cdc` — point out that it's **query/cursor-based capture**: Databricks periodically queries `orders`/`order_items` filtering on `updated_at > last_seen_value`, with history tracking Off (SCD1). `REPLICA IDENTITY FULL` and the publication are real, correct Postgres-side setup for logical replication in general, but this pipeline does not stream the WAL — make sure students don't walk away thinking it does, and flag that cursor-based capture cannot detect hard deletes), ~15 min GlobalMart table assignment (Lakeflow Connect vs Autoloader), ~15 min old mounting approach + storage-key problem, ~15 min Unity Catalog building blocks + live demo of `gbmart-ext-loc`. Do NOT have students create their own resources yet — that is HOL 1, immediately after this session.

## Section 1 — Recap: From Watermarks to a Managed Pipeline

In **ILT 1** we saw why a simple `WHERE updated_at > last_run` watermark falls apart:
- It misses **DELETEs** entirely — the row is just gone, nothing to filter on
- It only catches an UPDATE if the source table actually has an `updated_at` column
- At scale, re-scanning even the changed slice of a large table on a timer is wasteful

The gold-standard fix that captures every INSERT/UPDATE/DELETE with full fidelity — including hard deletes — is reading the **Write-Ahead Log (WAL)** directly through logical replication. We did that by hand with raw JDBC in ILT 1 (and go deeper in HOL 2) so the mechanics are not a mystery.

**Lakeflow Connect can run a database source in either of two connector modes:**

| Mode | How it detects changes | Catches DELETEs? |
|------|------------------------|-------------------|
| **Log-based (WAL / logical replication)** | Reads the database's write-ahead log directly, via a Postgres publication | Yes — a DELETE is a WAL event like any other, as long as `REPLICA IDENTITY FULL` is set |
| **Query-based (cursor column)** | Runs `WHERE <cursor_col> > last_value` on a schedule, same idea as a watermark | **No** — same blind spot as manual watermarking, because a deleted row simply stops matching the query |

> **GlobalMart's real, running pipeline (`orders_data_ingestion_cdc`) actually uses query-based / cursor capture — not log-based WAL streaming.** The cursor column is `updated_at` (refreshed by a Postgres trigger on every INSERT/UPDATE), and history tracking is **Off (SCD Type 1)** — the pipeline keeps only the latest row per key. On every run, Lakeflow Connect queries `orders`/`order_items` filtering `WHERE updated_at > last_seen_value` and upserts whatever comes back — the same idea as the manual watermark, just managed for you. Two things *were* set up on the Supabase side before the pipeline was built: `REPLICA IDENTITY FULL` on both `orders` and `order_items`, and a **publication** naming both tables. Both are real, correct setup steps — but they enable reliable Postgres logical-replication *infrastructure* in general; this particular pipeline doesn't use them to stream the WAL. Because it's cursor-based, **it cannot detect hard deletes** — if a row is deleted at the source, the pipeline has no way to notice, since a query filtering on `updated_at` only ever sees rows that still exist. Databricks still manages the connection, scheduling, retries, and schema handling for you — that's the real "managed pipeline" value — but DELETE capture is exactly the one thing this mode does not give you. The worked example in Section 3 walks through exactly what was set up.

```
Manual JDBC + WAL (ILT 1 / HOL 2)          Lakeflow Connect — query-based (GlobalMart's real pipeline)
──────────────────────────────────         ──────────────────────────────────────────────────────────
You open the JDBC connection                Databricks manages the connection
You create the replication slot             Not used by this pipeline — no replication slot here
You call pg_logical_slot_get_changes()       Databricks re-queries WHERE updated_at > last_seen_value
You parse INSERT/UPDATE/DELETE text          Databricks applies INSERT/UPDATE for you (not DELETE)
You write to Bronze yourself                 Lands as a governed, managed Unity Catalog table
Captures DELETEs, more code to maintain      Misses DELETEs, but far less code to maintain
```

## Section 2 — How Lakeflow Connect Works End to End

```
Step 1: Connection
   A "Connection" object in Unity Catalog stores how to reach the source database
   (host, port, credentials). Created once, reused by any number of pipelines.

Step 2: Ingestion Pipeline
   You pick a Connection, pick which schema/tables to sync, pick a connector mode
   (log-based WAL, or query-based cursor column), and pick a destination catalog +
   schema in Unity Catalog. Databricks handles the rest.

Step 3: First Run — Full Snapshot
   The pipeline reads every row currently in the source table(s) and lands
   them as the initial state of a Unity Catalog managed table.

Step 4: Every Run After — Incremental Sync
   Databricks reads only what changed since the last run and applies it to
   the destination table — without re-reading the whole source table.
   HOW it detects "what changed" depends on the connector mode from Step 2:
     - Log-based:   reads new WAL events via the replication slot (catches DELETEs)
     - Query-based: re-queries WHERE cursor_column > last_seen_value (misses DELETEs)

Step 5: Destination
   Lands as a Delta Streaming Table: <catalog>.<schema>.<table>
   Full lineage, access control, and audit — same as any other UC object.
   This is the table type Lakeflow Connect always creates, regardless of
   whether the connector mode is log-based or query-based.
```

### Why This Matters for `orders` and `order_items`

These two GlobalMart tables are **live, continuously changing** — order status moves from `pending` → `shipped` → `delivered`, line items get added or corrected. Re-reading the whole table on a timer (Autoloader-style full/incremental load) would either miss changes or waste enormous compute at GlobalMart's volume (~126,000 orders, ~377,000 order_items). Lakeflow Connect is the tool in the GlobalMart stack built for this shape of source — on every run it queries `WHERE updated_at > last_seen_value` and upserts just the rows that changed, without a hand-rolled watermark script, while Databricks manages the connection, scheduling, retries, and schema evolution for you. As covered in Section 1, this is the query-based/cursor mode: it correctly captures every INSERT and UPDATE, but — like any cursor-based watermark — it does **not** catch hard DELETEs.

## Section 3 — Worked Example: GlobalMart's Real `orders_data_ingestion_cdc` Pipeline

This is the actual, currently-running Lakeflow Connect pipeline behind GlobalMart's Bronze `orders`/`order_items` tables — not a hypothetical. We use it as the live worked example — in HOL 1 you will create your **own**, named after yourself, pointing at your own Supabase project.

### Step 0 — Supabase-Side Setup (Before the Connection Existed)

Before this pipeline could be built, a setup script was run **step by step in the Supabase SQL Editor** (not all at once — each step verified before moving to the next) to prepare `orders`/`order_items` for CDC:

```sql
-- 1. Create orders / order_items with an updated_at column
--    (both tables defined with their normal source columns, plus:)
--    updated_at TIMESTAMPTZ NOT NULL DEFAULT now()

-- 2. A trigger that refreshes updated_at on every INSERT/UPDATE
CREATE OR REPLACE FUNCTION set_updated_at() RETURNS trigger AS $$
BEGIN NEW.updated_at = now(); RETURN NEW; END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER trg_orders_updated_at
BEFORE INSERT OR UPDATE ON orders
FOR EACH ROW EXECUTE FUNCTION set_updated_at();
-- (same trigger pattern on order_items)

-- 3. REPLICA IDENTITY FULL — required for CDC to capture full row images
--    on UPDATE/DELETE, not just the primary key
ALTER TABLE orders       REPLICA IDENTITY FULL;
ALTER TABLE order_items  REPLICA IDENTITY FULL;

-- 4. A publication — exposes both tables to Lakeflow Connect over logical replication
CREATE PUBLICATION gbmart_pub FOR TABLE orders, order_items;

-- 5. Verify both tables are in the publication
SELECT * FROM pg_publication_tables WHERE pubname = 'gbmart_pub';
```

Data itself was loaded through **Supabase's Table Editor → Insert → Import data from CSV** — not row-by-row `INSERT` statements, because of volume: ~126,000 rows for `orders`, ~377,000 rows for `order_items`.

> **Note:** `REPLICA IDENTITY FULL` and the publication are real, correct Postgres-side setup — they're what make reliable logical replication *possible* for this database in general. As Step 2 below shows, though, GlobalMart's actual pipeline is configured in **query/cursor mode**, not log-based mode — so it doesn't end up using this setup to stream the WAL. This is standard, sensible practice either way: it means the option to switch to log-based capture later is available without redoing the Postgres-side work.

### Step 1 — The Connection (already created)

```
Databricks → Catalog → Connections → ecom_gbmart_conn

  Connection name : ecom_gbmart_conn
  Type            : PostgreSQL
  Host            : aws-0-ap-south-1.pooler.supabase.com
  Port            : 5432
```

This is a Unity Catalog Connection object — host/port/credentials are stored once, centrally, and referenced by name. No password ever appears in the pipeline definition itself. Once created, `orders` and `order_items` are browsable directly under the connection in Catalog Explorer — that's how you confirm both tables (and the publication from Step 0) are visible before building the pipeline.

### Step 2 — The Ingestion Pipeline (real spec, retrieved read-only from the workspace)

```
Pipeline name : orders_data_ingestion_cdc
Compute       : serverless          ← no cluster to size or manage
Connection    : ecom_gbmart_conn
Source type   : POSTGRESQL

  Source                                  Destination (Unity Catalog)
  ─────────────────────────────────       ──────────────────────────────
  postgres.globalmart.orders         →    gbmart.bronze.orders
  postgres.globalmart.order_items    →    gbmart.bronze.order_items

  Table configuration:
    primary_keys      : orderid           ← real source column, NO underscore
                         orderitemid       ← underscores (order_id) get added later, in Silver
    cursor_column      : updated_at       ← sequences which change is "latest" per row, query/cursor mode
    history_tracking   : Off (SCD Type 1) ← latest value per row only, no version history
    schedule           : none — skipped, pipeline is triggered manually
```

> **Notice:** the source column is `orderid`, not `order_id`. Bronze lands data exactly as the source names it. The clean, underscored `order_id` you'll see from Day 5 onward is a Silver-layer standardization — don't expect it in Bronze.

> **Why `updated_at` and not `OrderDate` as the cursor column?** The cursor column's job is to sequence which version of a row is the latest and to drive the `WHERE updated_at > last_seen_value` query Lakeflow Connect runs each time. `OrderDate` is set once, when the order is placed, and never changes again — so if an order's status later changes from `Shipped` to `Delivered`, `OrderDate` gives Lakeflow Connect no signal that anything happened. `updated_at` is refreshed by the trigger from Step 0 on every INSERT and UPDATE, so it always reflects the most recent change. Note that this cursor column only sequences and surfaces INSERT/UPDATE events — it has no way to represent a DELETE, since a deleted row simply has no `updated_at` left to query for.

> **One more per-table setting: history tracking.** The pipeline configuration also asks you to choose SCD1 (keep only the latest version of each row) or SCD2 (retain every past version as its own row). For a Bronze table that just needs to mirror the current source state — like `orders`/`order_items` here — SCD1 is the natural choice. SCD2 matters when the business genuinely needs history, like GlobalMart's address table (a customer moves from Mumbai to Bangalore and you need to keep both the old and new address). GlobalMart's real `orders_data_ingestion_cdc` pipeline is configured with history tracking **Off (SCD Type 1)** — exactly this choice.

> **Why no schedule?** For this build (and for practice/demo runs generally) the real pipeline was left without a schedule — skip it, don't add one, and trigger runs manually via **Start** whenever you want a sync. A production rollout would typically add a periodic schedule (e.g. hourly) once the pipeline is validated, but that's a separate, later decision from what's being taught here.

### First Run Result (illustrative row counts)

```
Status     : Completed
Tables     : orders, order_items
orders     : Upserted ~126,000 rows  ← full snapshot on first run
order_items: Upserted ~377,000 rows  ← full snapshot on first run
Catalog    : gbmart.bronze
```

### Every Run After — Insert and Update Sync; DELETE Does Not

```
Someone updates order O-10021, which sets its updated_at to now()
  → Pipeline is triggered manually (no schedule is set, see Step 2)
  → Upserted: 1     ← only that row, not the full table
  → Bronze reflects the new status within one pipeline run

Someone instead DELETEs a row in Supabase entirely
  → The row simply stops appearing in the query WHERE updated_at > last_seen_value
    — it hasn't been "updated," it no longer exists to match anything
  → Lakeflow Connect has no event to react to — nothing tells it a DELETE happened
  → The row is NOT removed from gbmart.bronze.orders — it silently goes stale
  → This is the real limitation of cursor-based capture: REPLICA IDENTITY FULL and the
    Step 0 publication are correct Postgres-side setup for logical replication in
    general, but this pipeline's cursor/query mode doesn't use them to catch deletes
```

> **Verify (SQL you'll run for real in HOL 1):**
> ```sql
> SELECT COUNT(*) FROM gbmart.bronze.orders;
> DESCRIBE EXTENDED gbmart.bronze.orders;
> ```

## Section 4 — GlobalMart's Full Ingestion Tool Assignment

Not every table needs Lakeflow Connect. Match the tool to the source's shape:

| Table | Source | Tool | Strategy | Why |
|-------|--------|------|----------|-----|
| `orders` | Postgres (Supabase) | **Lakeflow Connect** | Query/cursor-based capture (cursor: `updated_at`) | Continuously updated — INSERT and UPDATE captured every run; DELETEs are not (see Section 3) |
| `order_items` | Postgres (Supabase) | **Lakeflow Connect** | Query/cursor-based capture (cursor: `updated_at`) | Line items change with order status |
| `customers` | ADLS file drop | Autoloader | Incremental | New customers arrive daily — append + watermark |
| `products` | ADLS file drop | Autoloader | Full load | Catalog refreshed periodically — overwrite is safe |
| `addresses` | ADLS file drop | Autoloader | Incremental | Grows over time — append new rows |
| `payments` | ADLS file drop | Autoloader | Incremental | New payments arrive continuously |
| `payment_methods` | ADLS file drop | Autoloader | Full load | Small reference table — 4 rows |
| `returns` | ADLS file drop | Autoloader | Incremental | New returns arrive over time |

### Decision Rule

```
Is the source a live database with INSERT / UPDATE / DELETE happening continuously?
    → Lakeflow Connect
    (choose log-based/WAL if you must capture DELETEs; query-based/cursor if you don't —
     GlobalMart's real orders/order_items pipeline uses query-based/cursor and accepts
     the DELETE gap)

Is the source a file that lands in ADLS and, once landed, never changes?
    Does the table grow over time (new rows accumulate)?
        → Autoloader + Incremental Load (append + watermark)
    Is the table small and fully refreshed each time?
        → Autoloader + Full Load (overwrite)
```

---
## Section 5 — Old Approach: Mounting ADLS with `dbutils.fs.mount()`

Before Unity Catalog, teams gave a whole ADLS container a permanent path in the workspace, once, using a key or service principal:

```python
configs = {"fs.azure.account.key.STORAGE_ACCT.dfs.core.windows.net": "STORAGE_KEY"}

dbutils.fs.mount(
    source        = "abfss://container@STORAGE_ACCT.dfs.core.windows.net/",
    mount_point   = "/mnt/globalmart",
    extra_configs = configs
)

# From then on, every notebook in the workspace just does:
spark.read.csv("/mnt/globalmart/raw/customers.csv")
```

### Why This Doesn't Scale

| Problem | What It Means |
|---------|----------------|
| **Credentials baked in** | The key/service-principal secret is baked into the mount's cluster config — a security risk |
| **Workspace-level, not UC-governed** | A mount point is visible and usable by anyone in the whole workspace — Unity Catalog has no say over it |
| **No fine-grained access** | No way to grant one team read-only and another read-write, per user or role |
| **Bypasses governance entirely** | Mounts sit outside Unity Catalog's audit log, lineage, and permission model completely |

> **This is why nobody sets up new mounts anymore** — Storage Credentials + External Locations (Section 6 below) solve every one of these problems.

---
## Section 6 — Why Hardcoded Storage Keys Are Also a Problem

Since Day 1, every notebook that reads ADLS has started with this pattern — a different old approach from mounting, but the same underlying key-based risk:

```python
storage_account_name = "YOUR_STORAGE_ACCOUNT_NAME"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"   # ← 88-character secret

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
```

### Why This Doesn't Scale to a Real Team

| Problem | What It Means |
|---------|---------------|
| **Security risk** | Anyone who opens the notebook sees a key with full read/write access to the entire storage account |
| **Git exposure** | If committed to GitHub, the key stays in git history even after being deleted from the file |
| **Key rotation** | When the storage key is rotated (security policy), every notebook that hardcodes it breaks at once |
| **No audit trail** | Azure only sees "the key was used" — not which user, from which notebook |
| **All-or-nothing access** | Everyone with the key has full read/write. You cannot give one team read-only |

```
CURRENT FLOW (key-based):
  Notebook → spark.conf.set(key) → Spark driver → ADLS
                    ↑
              secret lives in plain text in the notebook cell

WHAT WE WANT:
  Notebook → abfss://... → Unity Catalog checks permission → ADLS
                                        ↑
                              no key anywhere — auth handled centrally
```

## Section 7 — The Two Unity Catalog Building Blocks

### Building Block 1 — Storage Credential

```
A Storage Credential is Databricks' identity card for Azure.

It wraps an Azure Managed Identity (the Databricks Access Connector) —
an Azure resource granted "Storage Blob Data Contributor" on your ADLS
storage account.

When Databricks needs to read/write ADLS, it uses this managed identity
to authenticate. No password, no key, nothing that can leak.
```

**Analogy:** the Storage Credential is an employee badge — it proves "this is who Databricks is" to Azure.

### Building Block 2 — External Location

```
An External Location is a registered ADLS path + which Storage Credential
to use for it.

  External Location: gbmart-ext-loc
    URL:                abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/
    Storage Credential: ecomprojectscredentials
```

**Analogy:** the External Location is the door the badge unlocks — it maps a specific room (an ADLS path) to a specific badge (a credential).

### The Full Auth Chain

```
Notebook uses abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/raw-data/
       ↓
Unity Catalog: "This path is covered by External Location 'gbmart-ext-loc'"
       ↓
Storage Credential: "Use managed identity 'ecomprojectscredentials'"
       ↓
Azure RBAC: "That managed identity has Storage Blob Data Contributor on ecomadlsdata"
       ↓
ADLS Gen2: access granted — data returned

At no point does a key appear anywhere in the notebook — and unlike a mount,
this whole chain is Unity Catalog-governed: auditable, and grantable per user/role.
```

## Section 8 — Worked Example: GlobalMart's Real External Location

Both objects already exist in this workspace, backing the real GlobalMart Bronze pipeline. We use them as the live worked example — in HOL 1 you will create your **own** storage credential + external location, named after yourself, pointing at your **own** storage account from Day 1.

```
Databricks → Catalog icon → External Data → Storage Credentials
  ecomprojectscredentials          Azure Managed Identity     Active

Databricks → Catalog icon → External Data → External Locations
  gbmart-ext-loc
    URL                : abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/
    Storage credential  : ecomprojectscredentials
```

This is the exact location the Autoloader Bronze notebooks (Day 3 / Day 4) read from — the base path `.../ecom-gbmart-data@ecomadlsdata.../raw-data` plus a subfolder per source (`customers/`, `products/`, `payments/`, etc.).

### Before vs After — What Changes in Every Notebook

**BEFORE (key-based auth, or a workspace-level mount):**
```python
storage_account_name = "YOUR_STORAGE_ACCOUNT_NAME"
container_name       = "YOUR_CONTAINER_NAME"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"   # ← secret in the notebook

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
```

**AFTER (Unity Catalog external location):**
```python
# No setup cell needed — Unity Catalog handles authentication via the
# External Location. The path is just a string, not a secret.
base_path = "abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/raw-data"
```

### Verify in SQL

```sql
SHOW EXTERNAL LOCATIONS;
DESCRIBE EXTERNAL LOCATION `gbmart-ext-loc`;

-- Grant access to a teammate — no key sharing required:
GRANT READ FILES ON EXTERNAL LOCATION `gbmart-ext-loc` TO `teammate@company.com`;
```

> **In HOL 1** you will create your own Access Connector–backed Storage Credential and External Location against your own Day-1 storage account, run `dbutils.fs.ls()` against it with zero keys in the notebook, and then build a Lakeflow Connect pipeline of your own against your own Supabase project. The `gbmart-ext-loc` shown above stays the shared, real location the rest of the course's Bronze/Silver/Gold pipeline actually reads from — your personal one is for practicing the skill.

---
## Recap

| Topic | Key Takeaway |
|-------|--------------|
| Lakeflow Connect | Managed Databricks pipeline; can run log-based (WAL) or query-based (cursor column) — GlobalMart's real pipeline uses **query-based/cursor capture** (cursor column `updated_at`, history tracking Off/SCD1), not log-based WAL streaming |
| Supabase-side setup | `REPLICA IDENTITY FULL` + a publication on `orders`/`order_items` — real, correct prerequisites for reliable logical replication in general, but GlobalMart's actual pipeline doesn't use them to stream the WAL (Step 0 of the worked example) |
| Connection | Stores how to reach the source DB — created once, reused by pipelines (`ecom_gbmart_conn`) |
| Ingestion Pipeline | Full snapshot on first run; every run after re-queries `WHERE updated_at > last_seen_value` and upserts INSERT/UPDATE — `updated_at` is the cursor column, and DELETEs are **not** captured |
| History tracking | Per-table SCD1 (latest state only) vs SCD2 (retain every past version) — GlobalMart's real pipeline uses SCD1; SCD2 is for genuine history needs (e.g. address changes). Neither setting captures a DELETE the pipeline never observed |
| Pipeline schedule | GlobalMart's real pipeline has no schedule — it's triggered manually. A periodic schedule (e.g. hourly) is a separate, later production decision |
| GlobalMart split | `orders` / `order_items` → Lakeflow Connect (query-based/cursor). Everything else → Autoloader |
| Mounting (old way) | `dbutils.fs.mount()` bakes credentials into cluster config, is workspace-level (not UC-governed), and gives no per-user/role access control |
| Storage Credential | Databricks' managed-identity "badge" for talking to Azure — no key |
| External Location | Maps an ADLS path to the credential that's allowed to access it |
| Real example | `gbmart-ext-loc` → `ecomprojectscredentials` → `ecomadlsdata` storage account |
| Governance win | Grant/revoke per user, full audit log, no key rotation to chase across notebooks |

---

## What Comes Next

| Session | Topic |
|---------|-------|
| **HOL 1 (next)** | Hands-on — create your own Storage Credential + External Location, and your own Lakeflow Connect pipeline for `orders` / `order_items` |
| **HOL 2** | Hands-on — WAL/replication-slot mechanics directly via JDBC (the log-based mode Lakeflow Connect *could* run, but doesn't for GlobalMart's real `orders`/`order_items` pipeline — done by hand this time) |
| **ILT 3 / HOL 3** | Code Versioning — Databricks Repos + GitHub |

---

**INSTRUCTOR NOTE:**
Closing check:
1. *'What does a Storage Credential actually contain?'* (Nothing secret you can see — it wraps an Azure Managed Identity via the Access Connector.)
2. *'What's wrong with a mount point that a Storage Credential + External Location fixes?'* (A mount is workspace-level, not governed by Unity Catalog, bakes credentials into cluster config, and gives no per-user/role access control — Unity Catalog's approach fixes all four.)
3. *'If GlobalMart rotates the ADLS storage key today, what breaks?'* (Nothing — Unity Catalog external locations don't use the account key at all.)
4. *'Why is `orders` on Lakeflow Connect and `products` on Autoloader?'* (`orders` changes continuously with INSERT/UPDATE/DELETE in a live database; `products` arrives as files that don't change once landed.)
5. *'If someone hard-deletes a row in the source `orders` table, will it disappear from `gbmart.bronze.orders`?'* (**No** — the real pipeline is query/cursor-based, not log-based. It queries `WHERE updated_at > last_seen_value`, and a deleted row simply has nothing left to match — Lakeflow Connect never sees a DELETE event, so the row stays in Bronze, stale. `REPLICA IDENTITY FULL` and the Step 0 publication are real, correct Postgres-side setup for logical replication in general, but this pipeline doesn't use them to stream the WAL — that's the log-based mode Lakeflow Connect *could* run, not the mode actually configured here. Only a true log-based/WAL connector, reading the publication directly, would catch the DELETE.)